# 🎙️ InfiniteTalk (Wan2.1) Web 视频对口型工作室

基于 **Wan2.1 (14B) + InfiniteTalk + LightX2V 蒸馏加速**，针对 Google Colab 优化。

> 💡 **推荐硬件**：**NVIDIA A100**（10秒视频仅需 2~3 分钟）；亦可在 **NVIDIA L4**（约 15 分钟）经济运行。
> 
> 📦 **代码仓库**：[https://github.com/ajaxsys/colab-lipsync](https://github.com/ajaxsys/colab-lipsync)

In [ ]:
# 1. 检查当前显卡规格
!nvidia-smi

In [ ]:
# 2. 克隆项目仓库并安装核心运行环境
import os

# 克隆主仓库
!git clone https://github.com/ajaxsys/colab-lipsync.git /content/colab-lipsync || (cd /content/colab-lipsync && git pull)
# 克隆 InfiniteTalk 依赖库
!git clone https://github.com/MeiGen-AI/InfiniteTalk.git /content/InfiniteTalk || true

# 安装依赖
%cd /content/colab-lipsync
!pip install -q -r requirements.txt

# 安装 Cloudflared 用于公网快速穿透
!if ! command -v cloudflared &> /dev/null; then wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb; fi
print("✅ 基础环境安装完成！")

In [ ]:
# 3. 准备模型权重 (支持从 Google Drive 秒级加载，或自动下载 29GB 完整模型)
import os

# 自动挂载 Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception as e:
    print(f"Drive 挂载提示: {e}")

gdrive_model_dir = "/content/drive/MyDrive/colab-web-model"
local_model_dir = "/content/models"

# 检查是否挂载了 Google Drive 且存在已备份的模型
if os.path.exists(f"{gdrive_model_dir}/dit/Wan2_1-I2V-14B-480P_fp8_e4m3fn.safetensors"):
    print("⚡ 检测到 Google Drive 备份模型，正在建立秒级软链接...")
    if not os.path.exists(local_model_dir):
        os.symlink(gdrive_model_dir, local_model_dir)
    print("🎉 模型秒级准备就绪，跳过 29GB 重复下载！")
else:
    print("📥 开始自动从 Hugging Face / 官方源下载完整权重 (~29GB)... (若已有备份请确认 Drive 授权)")
    !python download_models.py

In [ ]:
# 4. 启动 Web 工作室与 Cloudflare 公网隧道
import subprocess, time, re

# 终止可能残留的旧进程
!pkill -9 -f "app.py" || true
!pkill -9 -f "cloudflared" || true

print("🚀 正在启动 Web 服务...")
web_proc = subprocess.Popen(["python3", "-u", "app.py"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

time.sleep(3)
print("🌐 正在生成公网隧道链接...")
cf_proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:7860"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

tunnel_url = None
for line in cf_proc.stdout:
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        break

print("\n" + "="*60)
if tunnel_url:
    print(f"🎉 Web 界面已成功上线！点击下方链接打开：\n👉 {tunnel_url}?__theme=light")
else:
    print("⚠️ 隧道链接生成中，请在终端查看日志")
print("="*60 + "\n")

--- 
### 💾 附：Google Drive 模型备份 (只需执行一次)
下载完 29GB 模型后，运行下方单元格将模型备份到您的 Google Drive `colab-web-model` 文件夹。
下次启动无论换任何 Colab 机器（包括 A100），都可直接免下载秒级载入！

In [ ]:
# (可选) 挂载 Google Drive 并备份模型
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/colab-web-model
print("📦 开始同步模型文件到 Google Drive (约 29GB)... 请稍候")
!rsync -avP --delete /content/models/ /content/drive/MyDrive/colab-web-model/
print("🎉 备份完成！下次开启 Colab 可直接秒级挂载，无需重复下载。")